# Layer 06 - Chatbot RAG Evaluation

Companion to `05_chatbot_e2e_verification.ipynb` (which is a *behavioural* test) and to the legacy `05_chatbot/06_propertylens_rag_eval.ipynb` (which evaluated the retired Pinecone RAG). This notebook computes the **three classical RAG metrics** for the *current* `/api/property-search-chat` endpoint:

| Metric | What it measures | Why it matters here |
|---|---|---|
| **Hit Rate @ 5 + MRR** | Did the chatbot's top-5 historical-comparable rows surface any keyword we'd expect to see (town, flat type, school)? | Direct test of the Cypher retrieval layer fronted by the LLM. Low score = the LLM is extracting filters that exclude the right comps. |
| **Faithfulness** (LLM judge, 1-5) | Are the claims in the chatbot's narrative paragraph grounded in the row table above it, or did the LLM invent addresses / years / prices? | Direct test of today's anti-hallucination framing fix. 5 = fully grounded; 1 = significant invention. |
| **Answer Relevance** (LLM judge, 1-5) | Does the chatbot's response actually address the question, or is it off-topic / a non-answer? | Catches cases where retrieval succeeded but the LLM wrote a generic paragraph that doesn't help the user. |

**Key adaptation vs the legacy 06**: that notebook ran the Pinecone retrieval pipeline directly. This notebook hits the live `/api/property-search-chat` HTTP endpoint and parses the SSE response — same pattern as `05`. The chatbot's `### Historical comparables` row table is what we score for Hit Rate; the LLM narrative below is what we score for Faithfulness.

Pre-conditions:
- Backend on `:8000` (with today's framing + shortlist + school-cross-ref changes)
- Ollama with `gemma3` reachable on `:11434` (used both as the chatbot's Stage 3 generator AND as the LLM judge — see *Known limitation* at the bottom)
- Demo user `bhuvesh` / `1234`

Runtime: ~3-5 min (~25 chatbot calls + ~50 judge calls — judges are smaller / faster than the full chatbot Stage 3).

In [1]:
from __future__ import annotations

import json
import re
import sys
import time
from pathlib import Path

import pandas as pd
import requests
from IPython.display import display

API_BASE = 'http://localhost:8000'
OLLAMA_BASE = 'http://127.0.0.1:11434'
JUDGE_MODEL = 'gemma3'


def login(username: str, password: str = '1234') -> str:
    r = requests.post(
        f'{API_BASE}/api/auth/login',
        json={'username': username, 'password': password}, timeout=15,
    )
    r.raise_for_status()
    return r.json()['access_token']


JWT_BHUVESH = login('bhuvesh')
JWT_USER    = login('user')
print(f'Logged in: bhuvesh ({len(JWT_BHUVESH)}c), user ({len(JWT_USER)}c)')

_DECODE = lambda s: s.replace('\\\\', '\\').replace('\\n', '\n')


def post_chat(message: str, *, jwt: str | None = None, timeout_s: int = 60) -> dict:
    headers = {'Content-Type': 'application/json'}
    if jwt:
        headers['Authorization'] = f'Bearer {jwt}'
    t0 = time.perf_counter()
    r = requests.post(
        f'{API_BASE}/api/property-search-chat', headers=headers,
        json={'message': message, 'history': []}, stream=True, timeout=timeout_s,
    )
    r.raise_for_status()
    text_parts: list[str] = []
    params: dict | None = None
    for raw in r.iter_lines(decode_unicode=True):
        if not raw or not raw.startswith('data:'):
            continue
        body = raw[5:].strip()
        if body == '[DONE]':
            break
        if body.startswith('[LOG]') or body.startswith('[STATUS]'):
            continue
        if body.startswith('[PARAMS]') and body.endswith('[/PARAMS]'):
            try:
                params = json.loads(body[len('[PARAMS]'):-len('[/PARAMS]')])
            except Exception:
                pass
            continue
        text_parts.append(_DECODE(body))
    return {
        'markdown': '\n'.join(text_parts).strip(),
        'params': params or {},
        'latency_ms': int((time.perf_counter() - t0) * 1000),
    }


# Row format (current chatbot, today's framing):
#   - **1. 151 BISHAN ST 11** -- _sold 2020 for $420,000_  \n
#     BISHAN . 4 ROOM . 93 sqm . lease 68y . MRT 1098m . near PEI CHUN PUBLIC SCHOOL (0.9 km)
#
# Two-pass parse: split the section on row markers, then extract fields per
# row chunk. Single-regex parsing struggles because the optional
# `_sold YYYY for $X_` group interacts badly with the non-greedy filler.
_ROW_SPLIT_RE = re.compile(r'(?=^-\s+\*\*\d+\.\s)', re.MULTILINE)
_ADDR_RE      = re.compile(r"\*\*\d+\.\s+([A-Z0-9][A-Z0-9 .'\-]+?)\*\*")
_PRICE_RE     = re.compile(r"_(?:sold\s+(\d{4})\s+for\s+\$([\d,]+)|last\s+sale\s+\$([\d,]+))_")
_META_RE      = re.compile(r"\n\s{2,}([^\n]+)")


def extract_rows(md: str) -> list[dict]:
    """Extract structured rows from the `### Historical comparables` section.

    Returns a list of {address, year, price, meta} dicts in the order they appear.
    The `meta` field is the indented second line (e.g. `BISHAN . 4 ROOM . 93 sqm . ...`)
    which carries flat_type / town / school keywords used for Hit Rate matching.
    """
    if not md:
        return []
    idx = md.find('### Historical comparables')
    section = md[idx:] if idx >= 0 else md
    rows: list[dict] = []
    for raw in _ROW_SPLIT_RE.split(section):
        if not raw.lstrip().startswith('- **'):
            continue
        # Truncate at the first paragraph break -- that's where the LLM
        # narrative begins, and we don't want to absorb its text into meta.
        chunk = raw.split('\n\n', 1)[0]
        am = _ADDR_RE.search(chunk)
        if not am:
            continue
        pm = _PRICE_RE.search(chunk)
        mm = _META_RE.search(chunk)
        year = pm.group(1) if pm else None
        price_str = (pm.group(2) or pm.group(3)) if pm else None
        rows.append({
            'address': am.group(1).strip(),
            'year': int(year) if year else None,
            'price': int(price_str.replace(',', '')) if price_str else None,
            'meta': mm.group(1).strip() if mm else '',
        })
    return rows


def split_table_and_narrative(md: str) -> tuple[str, str]:
    """Split the response into the row-table block (context) and the LLM narrative paragraph."""
    if not md:
        return '', ''
    idx = md.find('### Historical comparables')
    if idx < 0:
        return md, ''
    rest = md[idx:]
    paras = rest.split('\n\n')
    table_paras = []
    narrative_paras = []
    for p in paras:
        if p.startswith('###') or p.startswith('_Past resale') or p.lstrip().startswith('- '):
            table_paras.append(p)
        else:
            narrative_paras.append(p)
    return '\n\n'.join(table_paras), '\n\n'.join(narrative_paras).strip()


print('Setup helpers ready.')


Logged in: bhuvesh (127c), user (123c)
Setup helpers ready.


## Evaluation test set

25 labelled query-answer pairs across the chatbot's surfaces.

Each entry has:
- `query` — natural-language question to send
- `category` — buyer / seller / search / school / negative
- `expected_answer` — short reference answer used by the Faithfulness and Answer-Relevance judges (direction, not exact wording)
- `relevant_keywords` — words that should appear in at least one retrieved row's address/town/flat_type for Hit Rate to score
- `jwt` — which demo user's JWT to use (defaults to bhuvesh)

**How to extend:** add rows to `EVAL_SET`. The more entries the more reliable the aggregate scores.

In [2]:
EVAL_SET: list[dict] = [
    # ── Buyer — price comp queries (4) ──────────────────────────────────────
    {
        'query': 'Find a 4-room flat in Bishan under $900k near MRT',
        'category': 'buyer',
        'expected_answer': 'Recent Bishan 4-room comps under $900k include sales in the $400k-$700k range; price varies with floor area, lease and storey.',
        'relevant_keywords': ['BISHAN', '4 ROOM'],
    },
    {
        'query': 'Find a 3 room flat in Hougang under $500k',
        'category': 'buyer',
        'expected_answer': 'Hougang 3-room flats have sold under $500k in recent years; comps span various blocks and lease remaining.',
        'relevant_keywords': ['HOUGANG', '3 ROOM'],
    },
    {
        'query': 'Show me a 4 room flat in Bishan with at least 90 sqm, within 500m of MRT, and under $800k',
        'category': 'buyer',
        'expected_answer': 'Recent Bishan 4-room comps within 500m of MRT and under $800k include several blocks with 90+ sqm units.',
        'relevant_keywords': ['BISHAN', '4 ROOM'],
    },
    {
        'query': 'I want a quiet 4 room flat in Tampines with at least 100 sqm and at least 60 years lease left',
        'category': 'buyer',
        'expected_answer': 'Tampines 4-room comps with 100+ sqm and 60+ years lease are available across multiple blocks; expect older leases to match smaller floor areas.',
        'relevant_keywords': ['TAMPINES', '4 ROOM'],
    },
    # ── Seller — listing-price reference (4) ─────────────────────────────────
    {
        'query': 'What did 4-room flats in Tampines sell for recently?',
        'category': 'seller',
        'expected_answer': 'Recent Tampines 4-room transactions span roughly $500k-$800k depending on block, floor and lease remaining.',
        'relevant_keywords': ['TAMPINES', '4 ROOM'],
    },
    {
        'query': 'Show me 5-room flats in Punggol with at least 100 sqm',
        'category': 'seller',
        'expected_answer': 'Punggol 5-room flats around 100+ sqm have transacted in the $600k-$800k range in recent years.',
        'relevant_keywords': ['PUNGGOL', '5 ROOM'],
    },
    {
        'query': 'Recent 5-room sales in Sengkang under $750k',
        'category': 'seller',
        'expected_answer': 'Sengkang 5-room comps under $750k are concentrated in older blocks; newer BTOs often exceed this band.',
        'relevant_keywords': ['SENGKANG', '5 ROOM'],
    },
    {
        'query': 'What did 4-room flats in Bedok sell for recently?',
        'category': 'seller',
        'expected_answer': 'Bedok 4-room flats have transacted broadly between $450k and $650k in recent years.',
        'relevant_keywords': ['BEDOK', '4 ROOM'],
    },
    # ── Search — broad criteria (3) ──────────────────────────────────────────
    {
        'query': 'Recommend a 4-room flat with the best access to top primary schools',
        'category': 'search',
        'expected_answer': '4-room comps near famous primary schools tend to cluster in mature estates near Bishan, Hougang, Tampines and central towns.',
        'relevant_keywords': ['4 ROOM'],
    },
    {
        'query': 'Find a 4 room flat in Tampines with famous school access under $750k',
        'category': 'search',
        'expected_answer': 'Tampines 4-room comps with a famous primary school within 1km include several blocks under $750k.',
        'relevant_keywords': ['TAMPINES', '4 ROOM'],
    },
    {
        'query': 'Find a 5 room flat in Bishan near MRT under $1,000,000',
        'category': 'search',
        'expected_answer': 'Bishan 5-room comps under $1M near MRT include several blocks; price reflects floor area, storey and lease remaining.',
        'relevant_keywords': ['BISHAN', '5 ROOM'],
    },
    # ── School graph (5) ─────────────────────────────────────────────────────
    {
        'query': 'Show me properties near POI CHING SCHOOL',
        'category': 'school',
        'expected_answer': 'POI CHING SCHOOL is in Tampines; the closest historical comps are on Tampines Street 71 and nearby blocks.',
        'relevant_keywords': ['TAMPINES'],
    },
    {
        'query': 'Show me flats near NANYANG PRIMARY SCHOOL',
        'category': 'school',
        'expected_answer': 'NANYANG PRIMARY SCHOOL is in Bukit Timah; nearby historical comps are on Queens Road and Farrer Road.',
        'relevant_keywords': ['BUKIT TIMAH'],
    },
    {
        'query': 'Show me flats near MAHA BODHI SCHOOL',
        'category': 'school',
        'expected_answer': 'MAHA BODHI SCHOOL is in the Geylang/Ubi area; nearby historical comps include Ubi Avenue blocks.',
        'relevant_keywords': ['GEYLANG', 'UBI'],
    },
    {
        'query': 'Show me flats near ROSYTH SCHOOL',
        'category': 'school',
        'expected_answer': 'ROSYTH SCHOOL is in the Hougang/Serangoon area; nearby historical comps are in Hougang and Serangoon blocks.',
        'relevant_keywords': ['HOUGANG', 'SERANGOON'],
    },
    {
        'query': 'Show me flats near HENRY PARK PRIMARY SCHOOL',
        'category': 'school',
        'expected_answer': 'HENRY PARK PRIMARY SCHOOL is in the Clementi/Bukit Timah area; nearby historical comps are in Clementi and adjacent blocks.',
        'relevant_keywords': ['CLEMENTI', 'BUKIT TIMAH'],
    },
    # ── Negative / framing-stress (4) ────────────────────────────────────────
    {
        'query': 'Find a 3 room flat in Bishan under $100k',
        'category': 'negative',
        'expected_answer': 'No matching historical sales — $100k is well below realistic market levels for any flat type in Bishan.',
        'relevant_keywords': ['BISHAN'],
    },
    {
        'query': 'Find a 5 room flat in Bukit Timah under $200k',
        'category': 'negative',
        'expected_answer': 'No matching historical sales — Bukit Timah 5-room flats transact far above $200k.',
        'relevant_keywords': ['BUKIT TIMAH'],
    },
    {
        'query': 'Where can I buy a 4-room in Bedok right now?',
        'category': 'negative',
        'expected_answer': 'These are historical comps, not current listings; for live inventory check PropertyGuru or 99.co.',
        'relevant_keywords': ['BEDOK'],
    },
    {
        'query': 'Are these properties for sale right now?',
        'category': 'negative',
        'expected_answer': 'No — the chatbot returns historical resale transactions, not current listings; visit PropertyGuru or 99.co for current inventory.',
        'relevant_keywords': [],
    },
    # ── Mixed / quiet+size+lease (5) ─────────────────────────────────────────
    {
        'query': 'I want a quiet 5 room flat in Tampines with at least 120 sqm and at least 70 years lease left',
        'category': 'search',
        'expected_answer': 'Tampines 5-room comps with 120+ sqm and 70+ years lease left include newer blocks distant from highways.',
        'relevant_keywords': ['TAMPINES', '5 ROOM'],
    },
    {
        'query': 'I want a quiet large flat with good lease left, not too expensive',
        'category': 'search',
        'expected_answer': 'Larger flats with long lease left in quieter towns away from highways include Sengkang, Punggol and parts of Tampines.',
        'relevant_keywords': [],
    },
    {
        'query': 'Find a 4-room flat in Bishan near famous schools under $900k',
        'category': 'search',
        'expected_answer': 'Bishan 4-room comps under $900k near famous primary schools include several blocks near AI TONG SCHOOL and PEI CHUN PUBLIC SCHOOL.',
        'relevant_keywords': ['BISHAN', '4 ROOM'],
    },
    {
        'query': 'Show me a 5 room flat in Sengkang within 600m of MRT under $700k',
        'category': 'search',
        'expected_answer': 'Sengkang 5-room comps within 600m of MRT and under $700k include several blocks across Compassvale and Sumang.',
        'relevant_keywords': ['SENGKANG', '5 ROOM'],
    },
    {
        'query': 'What did 4-room flats in Pasir Ris sell for recently?',
        'category': 'seller',
        'expected_answer': 'Recent Pasir Ris 4-room comps span roughly $450k-$650k depending on storey, lease and floor area.',
        'relevant_keywords': ['PASIR RIS', '4 ROOM'],
    },
]
print(f'Eval set: {len(EVAL_SET)} cases')
pd.DataFrame(EVAL_SET).groupby('category').size().to_frame('count')

Eval set: 25 cases


,count
category,
buyer,4
negative,4
school,5
search,7
seller,5


## Metric 1 — Hit Rate @ 5 + MRR

**What it measures:** for each query, did at least one of the chatbot's **top-5 historical-comparable rows** contain a relevant keyword (town, flat type, school catchment town)?

**Why it's the most important metric here:** it directly evaluates the retrieval stack — Stage-1 LLM intent extraction → Cypher → row formatting. A low hit rate means the LLM is misreading the question into filters that exclude the right comps.

**How it works:**
- Each eval entry has `relevant_keywords`. A row is a hit if its address / meta line contains any keyword (case-insensitive).
- For negative-result queries (where `relevant_keywords` covers what should NOT appear), we treat "no rows returned and a 'no matching' phrase present" as the correct outcome — recorded as `hit=True, rank=0` so MRR isn't penalised.
- **Hit Rate** = fraction of queries where at least one row is a hit.
- **MRR** = mean of `1/rank_of_first_hit` (0 if no hit). Penalises cases where the relevant comp only appears at rank 4 or 5.

In [3]:
_NEG_PHRASES = (
    'no matching historical sales found',
    "couldn't find matching properties",
    'couldn\u2019t find matching properties',
    'propertyguru', '99.co', 'current listing',
)


def row_is_hit(row: dict, keywords: list[str]) -> bool:
    if not keywords:
        return True
    haystack = (row['address'] + ' ' + row['meta']).upper()
    return any(kw.upper() in haystack for kw in keywords)


def evaluate_hit_rate(case: dict) -> dict:
    try:
        res = post_chat(case['query'], jwt=JWT_BHUVESH)
    except Exception as e:
        return {'query': case['query'][:50], 'category': case['category'],
                'hit': False, 'rank': None, 'mrr': 0.0,
                'rows_returned': 0, 'markdown': '', 'error': str(e), 'latency_ms': None}
    md = res['markdown']
    rows = extract_rows(md)
    md_lc = md.lower()
    is_negative = case['category'] == 'negative'
    looks_neg = any(p in md_lc for p in _NEG_PHRASES)

    if is_negative:
        # Pass if the chatbot honestly admits no matches OR redirects to live listings
        hit = looks_neg and (len(rows) == 0 or 'where can i buy' in case['query'].lower() or 'for sale' in case['query'].lower())
        hit_rank = 0 if hit else None
    else:
        hit_rank = None
        for r_i, row in enumerate(rows[:5], 1):
            if row_is_hit(row, case['relevant_keywords']):
                hit_rank = r_i
                break
        hit = hit_rank is not None

    mrr = (1.0 / hit_rank) if (hit_rank and hit_rank > 0) else (1.0 if hit_rank == 0 else 0.0)
    return {
        'query': case['query'][:55] + ('...' if len(case['query']) > 55 else ''),
        'category': case['category'],
        'rows_returned': len(rows),
        'hit': hit,
        'rank': hit_rank,
        'mrr': round(mrr, 3),
        'is_negative': is_negative,
        'latency_ms': res['latency_ms'],
        '_md_full': res['markdown'],   # kept for downstream metrics; trimmed in display
    }


hit_rows = [evaluate_hit_rate(c) for c in EVAL_SET]
hit_df = pd.DataFrame(hit_rows)
display(hit_df.drop(columns=['_md_full']).head(30))
summary = pd.DataFrame([{
    'total':     len(hit_df),
    'hit_rate':  round(hit_df['hit'].mean(), 4),
    'mrr':       round(hit_df['mrr'].mean(), 4),
    'avg_rows':  round(hit_df['rows_returned'].mean(), 1),
    'median_latency_ms': int(hit_df['latency_ms'].dropna().median()),
}])
display(summary)
display(
    hit_df.groupby('category')[['hit', 'mrr', 'rows_returned']].mean().round(3)
)

,query,category,rows_returned,hit,rank,mrr,is_negative,latency_ms
0,Find a 4-room flat in Bishan under $900k near MRT,buyer,5,True,1.0,1.0,False,12877
1,Find a 3 room flat in Hougang under $500k,buyer,5,True,1.0,1.0,False,9888
2,Show me a 4 room flat in Bishan with at least ...,buyer,3,True,1.0,1.0,False,10642
3,I want a quiet 4 room flat in Tampines with at...,buyer,5,True,1.0,1.0,False,9905
4,What did 4-room flats in Tampines sell for rec...,seller,5,True,1.0,1.0,False,14940
5,Show me 5-room flats in Punggol with at least ...,seller,0,False,NaN,0.0,False,3341
6,Recent 5-room sales in Sengkang under $750k,seller,5,True,1.0,1.0,False,7520
7,What did 4-room flats in Bedok sell for recently?,seller,5,True,1.0,1.0,False,11281
8,Recommend a 4-room flat with the best access t...,search,5,True,1.0,1.0,False,15088
9,Find a 4 room flat in Tampines with famous sch...,search,5,True,1.0,1.0,False,10343


,total,hit_rate,mrr,avg_rows,median_latency_ms
0,25,0.92,0.92,4.3,11281


,hit,mrr,rows_returned
category,,,
buyer,1.0,1.0,4.5
negative,1.0,1.0,2.5
school,0.8,0.8,5.0
search,1.0,1.0,5.0
seller,0.8,0.8,4.0


## Metric 2 — Faithfulness

**What it measures:** are the claims in the chatbot's narrative paragraph supported by the row table above it, or did the LLM invent addresses / years / prices?

**Why it matters most for the demo:** today's framing fix changed the Stage-3 prompt to forbid invention, but LLMs sometimes drift. This metric quantifies how often that drift happens.

**How it works:**
- Split each response into *table* (the `### Historical comparables` block) and *narrative* (the LLM paragraph below).
- Pass both to gemma3 as the judge with a strict prompt: *"are all factual claims in the narrative supported by the table?"* (5 = fully grounded, 1 = significant invention).
- For tool-only or shortlist responses (no table + no narrative split), faithfulness is N/A.

**Self-judge bias:** we use the same model (gemma3) as both generator and judge. Documented limitation; for production swap `JUDGE_MODEL` for a stronger judge (e.g. claude-haiku) or human-annotate a sample.

In [4]:
def _ollama_chat(prompt: str, *, model: str = JUDGE_MODEL, timeout_s: int = 60) -> str:
    r = requests.post(
        f'{OLLAMA_BASE}/api/chat',
        json={
            'model': model,
            'messages': [{'role': 'user', 'content': prompt}],
            'stream': False,
            'options': {'temperature': 0.1},
        },
        timeout=timeout_s,
    )
    r.raise_for_status()
    j = r.json() or {}
    return ((j.get('message') or {}).get('content') or '').strip()


def _parse_score(raw: str, default: float = 3.0) -> tuple[float, str]:
    score_m = re.search(r'SCORE:\s*([1-5])', raw)
    reason_m = re.search(r'REASON:\s*(.+)', raw)
    score = float(score_m.group(1)) if score_m else default
    reason = (reason_m.group(1).strip() if reason_m else raw[:140])
    return score, reason


def judge_faithfulness(query: str, table_md: str, narrative: str) -> tuple[float, str]:
    if not narrative.strip() or not table_md.strip():
        return 0.0, 'N/A — no narrative or no table'
    prompt = f"""You are evaluating a Singapore HDB property chatbot.

QUESTION: {query}

RETRIEVED ROW TABLE (this is the only ground truth — past resale transactions):
{table_md[:1500]}

GENERATED NARRATIVE PARAGRAPH (claims here MUST be supported by the table above):
{narrative[:1500]}

Task: Score whether every factual claim in the narrative is supported by the table.
Specifically check addresses, years and prices mentioned in the narrative — they MUST appear in the table.
Ignore stylistic differences and rounding.

Score 1-5:
5 = every claim is directly supported by the table
4 = mostly grounded, one minor unsupported detail
3 = mix of grounded and unsupported claims
2 = significant claims (address/year/price) not in table
1 = narrative invents addresses or contradicts the table

Return ONLY this format:
SCORE: <number 1-5>
REASON: <one sentence>
"""
    raw = _ollama_chat(prompt)
    return _parse_score(raw)


faith_rows = []
for hit_row, case in zip(hit_rows, EVAL_SET):
    md = hit_row['_md_full']
    table, narr = split_table_and_narrative(md)
    if not narr or not table:
        faith_rows.append({'query': hit_row['query'], 'category': hit_row['category'],
                            'faithfulness': None, 'reason': 'no narrative / no table'})
        continue
    try:
        score, reason = judge_faithfulness(case['query'], table, narr)
        faith_rows.append({'query': hit_row['query'], 'category': hit_row['category'],
                            'faithfulness': score, 'reason': reason[:80]})
    except Exception as e:
        faith_rows.append({'query': hit_row['query'], 'category': hit_row['category'],
                            'faithfulness': None, 'reason': f'judge_err: {type(e).__name__}'})

faith_df = pd.DataFrame(faith_rows)
display(faith_df.head(30))
judged = faith_df['faithfulness'].dropna()
if len(judged):
    print(f'Faithfulness  mean={judged.mean():.2f}  median={judged.median():.1f}  n={len(judged)}/{len(faith_df)}')
    display(faith_df.dropna(subset=['faithfulness']).groupby('category')['faithfulness'].mean().round(2).to_frame())

,query,category,faithfulness,reason
0,Find a 4-room flat in Bishan under $900k near MRT,buyer,4.0,The narrative accurately identifies several sa...
1,Find a 3 room flat in Hougang under $500k,buyer,5.0,"All factual claims regarding addresses (6, 7, ..."
2,Show me a 4 room flat in Bishan with at least ...,buyer,4.0,The narrative accurately identifies the addres...
3,I want a quiet 4 room flat in Tampines with at...,buyer,5.0,All factual claims regarding addresses (340 Ta...
4,What did 4-room flats in Tampines sell for rec...,seller,5.0,All factual claims regarding addresses (858B T...
5,Show me 5-room flats in Punggol with at least ...,seller,NaN,no narrative / no table
6,Recent 5-room sales in Sengkang under $750k,seller,5.0,All factual claims regarding addresses (256 Co...
7,What did 4-room flats in Bedok sell for recently?,seller,5.0,All factual claims regarding addresses (87 Bed...
8,Recommend a 4-room flat with the best access t...,search,5.0,"All factual claims regarding addresses (213, 2..."
9,Find a 4 room flat in Tampines with famous sch...,search,3.0,The narrative mentions specific sale prices an...


Faithfulness  mean=4.50  median=5.0  n=22/25


,faithfulness
category,
buyer,4.5
negative,4.0
school,5.0
search,4.0
seller,5.0


## Metric 3 — Answer Relevance

**What it measures:** does the chatbot's response actually address the question, or is it off-topic / a non-answer?

**Why it matters:** even with perfect retrieval and faithfulness, the chatbot could write a generic paragraph that doesn't help the user (e.g. listing flats when the user asked about price ranges). This catches that.

**How it works:** gemma3 as judge again. Compares the response (table + narrative) to the `expected_answer` direction. 5 = directly addresses the question; 1 = off-topic or non-answer.

In [5]:
def judge_relevance(query: str, expected: str, actual: str) -> tuple[float, str]:
    prompt = f"""You are evaluating a Singapore HDB property chatbot.

QUESTION: {query}

REFERENCE ANSWER (direction, not exact wording):
{expected}

ACTUAL CHATBOT RESPONSE:
{actual[:2000]}

Task: Score whether the actual response addresses the question.
Use the reference only as a guide to expected direction — different wording is fine.
Penalise: non-answers, off-topic responses, refusing to answer when evidence is available, wrong verdict.
If the question asks about CURRENT listings and the response correctly redirects to PropertyGuru or 99.co, that's a CORRECT answer (score 5).

Score 1-5:
5 = directly and completely answers the question
4 = mostly answers with minor gaps
3 = partially answers, important aspect missing
2 = tangentially related but does not answer
1 = does not answer

Return ONLY this format:
SCORE: <number 1-5>
REASON: <one sentence>
"""
    raw = _ollama_chat(prompt)
    return _parse_score(raw)


rel_rows = []
for hit_row, case in zip(hit_rows, EVAL_SET):
    md = hit_row['_md_full']
    if not md.strip():
        rel_rows.append({'query': hit_row['query'], 'category': hit_row['category'],
                          'relevance': None, 'reason': 'empty response'})
        continue
    try:
        score, reason = judge_relevance(case['query'], case['expected_answer'], md)
        rel_rows.append({'query': hit_row['query'], 'category': hit_row['category'],
                          'relevance': score, 'reason': reason[:80]})
    except Exception as e:
        rel_rows.append({'query': hit_row['query'], 'category': hit_row['category'],
                          'relevance': None, 'reason': f'judge_err: {type(e).__name__}'})

rel_df = pd.DataFrame(rel_rows)
display(rel_df.head(30))
judged = rel_df['relevance'].dropna()
if len(judged):
    print(f'Relevance     mean={judged.mean():.2f}  median={judged.median():.1f}  n={len(judged)}/{len(rel_df)}')
    display(rel_df.dropna(subset=['relevance']).groupby('category')['relevance'].mean().round(2).to_frame())

,query,category,relevance,reason
0,Find a 4-room flat in Bishan under $900k near MRT,buyer,4.0,The chatbot provides a list of historical sale...
1,Find a 3 room flat in Hougang under $500k,buyer,5.0,The chatbot successfully identified and presen...
2,Show me a 4 room flat in Bishan with at least ...,buyer,4.0,The chatbot provides relevant historical data ...
3,I want a quiet 4 room flat in Tampines with at...,buyer,4.0,The chatbot provides several relevant historic...
4,What did 4-room flats in Tampines sell for rec...,seller,4.0,The chatbot provides several recent sales pric...
5,Show me 5-room flats in Punggol with at least ...,seller,2.0,The chatbot failed to provide any relevant inf...
6,Recent 5-room sales in Sengkang under $750k,seller,4.0,The chatbot provides relevant historical sales...
7,What did 4-room flats in Bedok sell for recently?,seller,4.0,The chatbot provides several recent resale tra...
8,Recommend a 4-room flat with the best access t...,search,4.0,The chatbot provides a list of 4-room flats ne...
9,Find a 4 room flat in Tampines with famous sch...,search,4.0,The chatbot provides relevant historical compa...


Relevance     mean=4.12  median=4.0  n=25/25


,relevance
category,
buyer,4.25
negative,3.75
school,4.80
search,4.00
seller,3.80


## Summary table

All three metrics per query, plus aggregate means and per-category breakdown.

In [6]:
summary_df = pd.DataFrame({
    'query':         hit_df['query'],
    'category':      hit_df['category'],
    'rows_returned': hit_df['rows_returned'],
    'hit':           hit_df['hit'],
    'rank':          hit_df['rank'],
    'mrr':           hit_df['mrr'],
    'faithfulness':  faith_df['faithfulness'],
    'relevance':     rel_df['relevance'],
    'latency_ms':    hit_df['latency_ms'],
})
display(summary_df)

agg = pd.DataFrame([{
    'cases':              len(summary_df),
    'hit_rate@5':         round(summary_df['hit'].mean(), 4),
    'mrr':                round(summary_df['mrr'].mean(), 4),
    'faithfulness_mean':  round(summary_df['faithfulness'].dropna().mean(), 2) if summary_df['faithfulness'].notna().any() else None,
    'relevance_mean':     round(summary_df['relevance'].dropna().mean(), 2) if summary_df['relevance'].notna().any() else None,
    'median_latency_ms':  int(summary_df['latency_ms'].dropna().median()),
    'p95_latency_ms':     int(summary_df['latency_ms'].dropna().quantile(0.95)),
}])
display(agg)
display(
    summary_df.groupby('category')
      .agg({'hit': 'mean', 'mrr': 'mean',
            'faithfulness': lambda s: round(s.dropna().mean(), 2) if s.notna().any() else None,
            'relevance':    lambda s: round(s.dropna().mean(), 2) if s.notna().any() else None})
      .round(3)
)

,query,category,rows_returned,hit,rank,mrr,faithfulness,relevance,latency_ms
0,Find a 4-room flat in Bishan under $900k near MRT,buyer,5,True,1.0,1.0,4.0,4.0,12877
1,Find a 3 room flat in Hougang under $500k,buyer,5,True,1.0,1.0,5.0,5.0,9888
2,Show me a 4 room flat in Bishan with at least ...,buyer,3,True,1.0,1.0,4.0,4.0,10642
3,I want a quiet 4 room flat in Tampines with at...,buyer,5,True,1.0,1.0,5.0,4.0,9905
4,What did 4-room flats in Tampines sell for rec...,seller,5,True,1.0,1.0,5.0,4.0,14940
5,Show me 5-room flats in Punggol with at least ...,seller,0,False,NaN,0.0,NaN,2.0,3341
6,Recent 5-room sales in Sengkang under $750k,seller,5,True,1.0,1.0,5.0,4.0,7520
7,What did 4-room flats in Bedok sell for recently?,seller,5,True,1.0,1.0,5.0,4.0,11281
8,Recommend a 4-room flat with the best access t...,search,5,True,1.0,1.0,5.0,4.0,15088
9,Find a 4 room flat in Tampines with famous sch...,search,5,True,1.0,1.0,3.0,4.0,10343


,cases,hit_rate@5,mrr,faithfulness_mean,relevance_mean,median_latency_ms,p95_latency_ms
0,25,0.92,0.92,4.5,4.12,11281,17712


,hit,mrr,faithfulness,relevance
category,,,,
buyer,1.0,1.0,4.5,4.25
negative,1.0,1.0,4.0,3.75
school,0.8,0.8,5.0,4.80
search,1.0,1.0,4.0,4.00
seller,0.8,0.8,5.0,3.80


## How to interpret the scores

### Hit Rate @ 5
| Score | Interpretation | Action |
|-------|---------------|--------|
| > 0.85 | Retrieval is healthy | Move on to faithfulness / relevance |
| 0.65 – 0.85 | LLM Stage-1 occasionally extracts overly tight filters | Lower Ollama temperature, add few-shot examples for failing categories |
| < 0.65 | Retrieval is broken | Check `_STAGE1_SYSTEM` prompt, check Cypher templates, verify Neo4j has the expected nodes |

### MRR
| Score | Interpretation |
|-------|---------------|
| > 0.7 | Right answers usually at rank 1-2 |
| 0.4 – 0.7 | Right answers often at rank 3-5; secondary weights drift |
| < 0.4 | Right answers rarely in top 3; ranking is misordered or many misses |

### Faithfulness
| Score | Interpretation | Action |
|-------|---------------|--------|
| 4.0 – 5.0 | Narrative stays grounded in the table | Production-ready |
| 3.0 – 4.0 | Occasional unsupported addresses / years | Tighten Stage-3 system prompt; add explicit "quote only addresses present in the rows" |
| < 3.0 | Significant invention | Drop Stage 3 entirely — the table is the answer; LLM paraphrase adds latency + risk |

### Answer Relevance
| Score | Interpretation | Action |
|-------|---------------|--------|
| 4.0 – 5.0 | Chatbot addresses questions well | Good |
| 3.0 – 4.0 | Some off-topic answers | Check Stage-1 intent extraction (Section 2 of `05`) |
| < 3.0 | Frequent dodging or off-topic | Improve `_STAGE3_SYSTEM` prompt; add intent-specific instructions |

### Known limitations
- **Self-judge bias**: gemma3 evaluates its own outputs (Stage-3 generator and judge are the same model). Real numbers are likely 0.3-0.5 points lower with a stronger external judge. Documented; swap `JUDGE_MODEL` for production.
- **Hit Rate keywords are coarse**: a chunk in TAMPINES counts as a hit for any TAMPINES query regardless of flat type / lease / price. Add stricter `relevant_keywords` lists if you need finer signal.
- **Latency dominated by Stage 1 + Stage 3 LLM** (~5-7s per case). Judge calls add another ~2-3s each. Total runtime ~10-15 min for the full eval (3 metrics × 25 cases × 2-7s).
- **Negative-result cases** are deliberately scored as `hit` when the chatbot honestly says "no matches" — that's the desired behaviour, not a retrieval failure.